# PyTorch Foundation · Day 01

## 틀린 예측이 어떻게 실제로 나아지는가

오늘의 목표는 PyTorch API를 많이 외우는 것이 아님.  
`데이터 표현 → shape 읽기 → prediction → loss → gradient → parameter update → 새 prediction`을 실제 변화로 설명하는 것이 목표임.


![Day 01 전체 학습 지도](assets/01_learning_loop.svg)


## Prelude · 컴퓨터는 틀린 예측에서 어떻게 나아질까?

입력 `2`를 받은 현재 규칙이 `6`을 예측했고 실제값은 `10`이라고 가정함.

1. 예측을 만듦
2. 실제값과 얼마나 다른지 측정함
3. 내부 값을 조금 바꿈
4. 같은 입력으로 다시 예측함

오늘은 이 반복을 가장 작은 코드로 직접 확인함.


In [1]:
input_value = 2.0
current_weight = 3.0
actual_value = 10.0

prediction = current_weight * input_value
difference = prediction - actual_value

print("입력:", input_value)
print("현재 예측:", prediction)
print("실제값:", actual_value)
print("차이:", difference)


입력: 2.0
현재 예측: 6.0
실제값: 10.0
차이: -4.0


### 관찰

- 현재 규칙은 예측을 만들 수 있음
- 예측과 실제값의 차이도 계산할 수 있음
- 아직 내부 값 `current_weight`를 어느 방향으로 얼마나 바꿀지는 모름

이 빈칸을 오늘 하나씩 채움.


## 1. 데이터를 PyTorch에서는 어떻게 표현할까?

모델은 Python 목록을 의미만으로 이해하지 못함. 계산할 값과 그 값의 모양·종류·위치를 함께 다룰 그릇이 필요함. PyTorch에서는 그 그릇을 **Tensor**라고 부름.


In [2]:
import torch

print("PyTorch 버전:", torch.__version__)
print("오늘의 기본 실행 위치:", torch.device("cpu"))


PyTorch 버전: 2.14.0
오늘의 기본 실행 위치: cpu


### Tensor를 만들 때 먼저 묻는 세 가지

함수 이름부터 외우지 않고 필요한 값의 상태를 먼저 판단함.

1. **값을 이미 알고 있는가?** → 직접 값을 넣거나 일정한 범위를 만듦
2. **구조만 먼저 준비하는가?** → 0 또는 1로 shape를 만듦
3. **임의의 시작값이 필요한가?** → 난수를 사용하되 분포와 seed를 확인함

아래 셀은 여섯 함수를 한 번씩만 비교함. 실행 전에 각 결과의 shape를 예상해봄.


In [3]:
torch.manual_seed(7)

known_values = torch.tensor([2.0, 4.0, 6.0])
zero_frame = torch.zeros(2, 3)
one_frame = torch.ones(2, 3)
ordered_values = torch.arange(0, 6, 2)
uniform_samples = torch.rand(2, 3)
normal_samples = torch.randn(2, 3)

created = {
    "직접 지정": known_values,
    "0으로 준비": zero_frame,
    "1로 준비": one_frame,
    "일정 간격": ordered_values,
    "0 이상 1 미만 난수": uniform_samples,
    "표준정규 계열 난수": normal_samples,
}

for label, value in created.items():
    print(f"{label:18s} shape={tuple(value.shape)}")
    print(value)


직접 지정              shape=(3,)
tensor([2., 4., 6.])
0으로 준비             shape=(2, 3)
tensor([[0., 0., 0.],
        [0., 0., 0.]])
1로 준비              shape=(2, 3)
tensor([[1., 1., 1.],
        [1., 1., 1.]])
일정 간격              shape=(3,)
tensor([0, 2, 4])
0 이상 1 미만 난수       shape=(2, 3)
tensor([[0.5349, 0.1988, 0.6592],
        [0.6569, 0.2328, 0.4251]])
표준정규 계열 난수         shape=(2, 3)
tensor([[-1.2514, -1.8841,  0.4457],
        [-0.7068, -1.5750, -0.6318]])


### 출력에서 확인할 것

- `torch.tensor`: 이미 알고 있는 값을 직접 담음
- `torch.zeros`, `torch.ones`: 원하는 shape를 각각 0과 1로 준비함
- `torch.arange`: 시작·끝·간격에 따라 순서가 있는 값을 만듦
- `torch.rand`: 0 이상 1 미만의 균등분포에서 표본을 만듦
- `torch.randn`: 평균 0, 표준편차 1인 표준정규분포 계열에서 표본을 만듦

`manual_seed(7)`은 같은 환경에서 난수 출력을 다시 확인하기 위한 설정임. 출력 숫자 자체는 암기하지 않음. 함수 선택 이유와 결과 shape를 한 문장씩 말해봄.


![scalar, vector, matrix, higher tensor](assets/02_tensor_shapes.svg)

숫자 하나도 Tensor가 될 수 있고, 여러 축을 가진 데이터도 Tensor가 될 수 있음. 핵심은 차원 이름 암기보다 **각 축의 의미를 읽는 것**임.


In [4]:
scalar = torch.tensor(7.0)
vector = torch.tensor([18.0, 21.0, 25.0])
matrix = torch.tensor([[18.0, 45.0, 0.0],
                       [21.0, 55.0, 1.0]])
mini_images = torch.arange(24, dtype=torch.float32).reshape(2, 3, 2, 2)

for name, value in {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "mini_images": mini_images,
}.items():
    print(f"{name:12s} shape={tuple(value.shape)}, ndim={value.ndim}")


scalar       shape=(), ndim=0
vector       shape=(3,), ndim=1
matrix       shape=(2, 3), ndim=2
mini_images  shape=(2, 3, 2, 2), ndim=4


### shape를 읽는 순서

- `[]`: 축이 없는 숫자 하나
- `[3]`: 원소 3개를 가진 축 하나
- `[2, 3]`: 첫 축 2개, 둘째 축 3개
- `[2, 3, 2, 2]`: 예시에서는 `[sample, channel, height, width]`

`[2, 3]` 자체만 말하지 않고, 가능하면 `관측 2개 × 특성 3개`처럼 의미까지 말함.


![semantic axes](assets/03_semantic_axes.svg)


### 같은 shape, 다른 semantic axes

shape의 숫자는 저장 구조를 말하지만 축의 이름까지 자동으로 말해주지는 않음. 같은 `(2, 3)`이라도 한 Tensor는 `학생 2명 × 과목 3개 점수`, 다른 Tensor는 `샘플 2개 × class score 3개`일 수 있음. 실행 전에 두 Tensor의 shape가 같은지, 같은 계산에 바로 넣어도 되는지 각각 예상해봄.


In [5]:
student_subject_scores = torch.tensor([[80.0, 90.0, 70.0],
                                       [75.0, 85.0, 95.0]])
sample_class_scores = torch.tensor([[2.1, 0.3, -0.4],
                                    [0.2, 1.8, 0.1]])

print("학생-과목 점수 shape:", tuple(student_subject_scores.shape))
print("샘플-class score shape:", tuple(sample_class_scores.shape))
print("shape만 같은가?:", student_subject_scores.shape == sample_class_scores.shape)


학생-과목 점수 shape: (2, 3)
샘플-class score shape: (2, 3)
shape만 같은가?: True


두 shape를 숫자로만 읽지 않고 각각 한 문장으로 적어봄.

- 첫 Tensor: 첫 axis는 ___, 둘째 axis는 ___임.
- 둘째 Tensor: 첫 axis는 ___, 둘째 axis는 ___임.
- 두 Tensor의 shape가 같아도 서로 빼는 계산의 의미를 자동으로 보장할 수 없는 이유는 ___임.

변수명만 보고 확정하기보다 데이터 설명과 이후 연산이 기대하는 계약까지 확인함.


In [6]:
temperatures = torch.tensor([18, 21, 25])
measurements = torch.tensor([[18.0, 45.0],
                             [21.0, 55.0],
                             [25.0, 60.0]])

print("temperatures:", temperatures.dtype, temperatures.device, tuple(temperatures.shape))
print("measurements:", measurements.dtype, measurements.device, tuple(measurements.shape))


temperatures: torch.int64 cpu (3,)
measurements: torch.float32 cpu (3, 2)


### dtype와 device

- `dtype`: 값의 종류임. 정수와 실수는 계산 역할이 다를 수 있음
- `device`: Tensor가 계산되는 위치임. 오늘 코드는 CPU만으로 완전히 실행됨
- Tensor끼리 계산할 때는 dtype과 device가 맞는지 확인하는 습관이 필요함

GPU 전체 파이프라인은 Day 2에서 다시 연결함.


### 짧은 실습 · axis 의미 말하기

`mini_images.shape == [2, 3, 2, 2]`를 다음 문장으로 읽어봄.

> 샘플 ___개, 샘플마다 채널 ___개, 각 채널의 높이 ___, 너비 ___임.

숫자를 채운 뒤 실제 shape와 비교함.


### 같은 숫자라도 dtype과 역할은 다를 수 있음

화면에 `0, 1, 2`가 보인다는 사실만으로 Tensor의 역할을 결정할 수 없음. 연속적인 측정값이면 실수 계산이 필요할 수 있고, 범주 번호라면 정수 인덱스 역할일 수 있음. 먼저 두 Tensor의 값과 shape가 비슷해 보이는지 확인하고, 실행 전에 dtype이 같을지 예상해봄.


In [7]:
measurement_values = torch.tensor([0.0, 1.0, 2.0])
category_ids = torch.tensor([0, 1, 2])

for name, value in {"measurement_values": measurement_values, "category_ids": category_ids}.items():
    print(f"{name:20s} values={value.tolist()}, shape={tuple(value.shape)}, dtype={value.dtype}")


measurement_values   values=[0.0, 1.0, 2.0], shape=(3,), dtype=torch.float32
category_ids         values=[0, 1, 2], shape=(3,), dtype=torch.int64


실행 결과를 보고 다음을 설명해봄.

1. 두 Tensor에서 같은 것은 무엇이고 다른 것은 무엇인가?
2. 소수점 표시가 아니라 dtype 출력이 필요한 이유는 무엇인가?
3. 역할을 확인하지 않고 모든 Tensor에 무조건 `.float()`를 붙이면 무엇을 숨길 수 있는가?

오늘은 dtype 변환 함수 목록으로 확장하지 않음. parameter·prediction·MSE는 실수 Tensor로 다룬다는 현재 계약만 분명히 함.


## 2. 원하는 부분을 어떻게 선택할까?

아래 데이터의 shape는 `[4, 3]`임. axis 0은 `관측 4개`, axis 1은 `온도·습도·비 여부`라는 `특성 3개`임.


In [8]:
weather = torch.tensor([
    [18.0, 45.0, 0.0],
    [21.0, 55.0, 1.0],
    [25.0, 60.0, 1.0],
    [20.0, 50.0, 0.0],
])
feature_names = ["온도", "습도", "비 여부"]
print(weather)
print("shape:", tuple(weather.shape))


tensor([[18., 45.,  0.],
        [21., 55.,  1.],
        [25., 60.,  1.],
        [20., 50.,  0.]])
shape: (4, 3)


In [9]:
print("두 번째 관측 전체:", weather[1])
print("모든 관측의 온도:", weather[:, 0])
print("앞의 두 관측, 온도와 습도:\n", weather[:2, :2])
print("비가 온 관측:\n", weather[weather[:, 2] == 1])


두 번째 관측 전체: tensor([21., 55.,  1.])
모든 관측의 온도: tensor([18., 21., 25., 20.])
앞의 두 관측, 온도와 습도:
 tensor([[18., 45.],
        [21., 55.]])
비가 온 관측:
 tensor([[21., 55.,  1.],
        [25., 60.,  1.]])


### 관찰

- `weather[1]`: axis 0에서 한 관측을 선택함
- `weather[:, 0]`: 모든 관측을 유지하고 axis 1에서 온도만 선택함
- `weather[:2, :2]`: 두 축에서 범위를 선택함
- 조건식은 `True`인 관측만 남김

선택 결과의 shape가 바뀌었다면 어떤 axis가 사라졌거나 유지됐는지 함께 읽음.


### 값이 비슷해도 axis가 유지될 수 있음

`weather[1]`과 `weather[1:2]`는 둘 다 두 번째 관측의 세 값을 가리킴. 그러나 정수 하나로 고르면 관측 axis가 사라지고, 범위로 고르면 길이 1인 관측 axis가 남음. 실행 전에 두 결과의 값과 shape를 각각 예상해봄.


In [10]:
one_observation = weather[1]
one_observation_batch = weather[1:2]

print("weather[1] 값:", one_observation)
print("weather[1] shape:", tuple(one_observation.shape))
print("weather[1:2] 값:\n", one_observation_batch)
print("weather[1:2] shape:", tuple(one_observation_batch.shape))


weather[1] 값: tensor([21., 55.,  1.])
weather[1] shape: (3,)
weather[1:2] 값:
 tensor([[21., 55.,  1.]])
weather[1:2] shape: (1, 3)


관찰 뒤 다음 문장을 완성해봄.

> 한 관측을 특성 vector로 사용할 때는 ___가 자연스러울 수 있고, batch 크기 1을 유지해야 하는 계산에서는 ___가 필요할 수 있음.

둘 중 하나가 언제나 옳다는 결론은 내리지 않음. 다음 연산이 `[feature]`를 기대하는지 `[batch, feature]`를 기대하는지 확인한 뒤 선택함.


### 짧은 실습 · 습도만 선택하기

**목적**: axis 의미를 이용해 slicing함.

`weather`에서 모든 관측의 습도만 선택하고 shape를 확인함.

<details><summary>Hint 1</summary>
모든 행은 유지하고, 특성 축에서 습도의 위치를 선택함.
</details>

<details><summary>Hint 2</summary>
형태는 <code>weather[:, 열_위치]</code>임.
</details>


In [ ]:
# 아래 줄을 수정해 모든 관측의 습도만 선택해봄
humidity_attempt = weather
print("현재 선택의 shape:", tuple(humidity_attempt.shape))
print(humidity_attempt)


In [ ]:
humidity_answer = weather[:, 1]
print(f'답의 shape: {tuple(humidity_answer.shape)}')
print(humidity_answer)

## 3. 같은 값을 다른 모양으로 어떻게 볼까?

모델 입력에 맞추기 위해 shape를 바꿀 때가 많음. 이때 값의 총개수는 유지되어야 하고, axis의 의미가 어떻게 바뀌는지 설명할 수 있어야 함.


In [15]:
sequence = torch.arange(24)
blocks = sequence.reshape(2, 3, 4)
a = sequence.reshape(2, 12)
flattened_per_sample = blocks.reshape(2, -1)
b = blocks.reshape(3, -1)
print(b)

print("원본:", tuple(sequence.shape), "원소 수:", sequence.numel())
print("blocks:", tuple(blocks.shape), "원소 수:", blocks.numel())
print("샘플별 펼침:", tuple(flattened_per_sample.shape), "원소 수:", flattened_per_sample.numel())


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23]])
원본: (24,) 원소 수: 24
blocks: (2, 3, 4) 원소 수: 24
샘플별 펼침: (2, 12) 원소 수: 24


### reshape 가능성과 의미 보존을 분리함

12개 값은 `(3, 4)`, `(4, 3)`, `(2, 6)`으로 모두 reshape 가능함. 원소 수 계약을 만족하기 때문임. 그러나 원본이 `학생 3명 × 문항 4개`라면 세 모양이 모두 같은 의미를 보존하는 것은 아님. 특히 `reshape(4, 3)`을 행과 열을 뒤집는 transpose와 같은 동작으로 부르면 안 됨.

실행 전에 `reshape(4, 3)`과 `T`의 값 배치가 같을지 예상해봄.


In [19]:
score_grid = torch.arange(1, 13).reshape(3, 4)
possible_but_reinterpreted = score_grid.reshape(4, 3)
axes_swapped = score_grid.T

print("원본 [학생=3, 문항=4]:\n", score_grid)
print("reshape(4, 3):\n", possible_but_reinterpreted)
print("transpose 결과:\n", axes_swapped)
print("두 결과의 값 배치가 같은가?:", torch.equal(possible_but_reinterpreted, axes_swapped))


원본 [학생=3, 문항=4]:
 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
reshape(4, 3):
 tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])
transpose 결과:
 tensor([[ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11],
        [ 4,  8, 12]])
두 결과의 값 배치가 같은가?: False


### 직접 확인하기 · 세 기준으로 shape 변경 판정하기

다음 후보를 `실행 가능 / 의미 보존 / 축 교환` 세 기준으로 나누어 설명해봄.

1. 학생 3명 × 문항 4개를 `reshape(3, 4)`로 유지함.
2. 같은 값을 `reshape(2, 6)`으로 바꿈.
3. 학생 축과 문항 축을 바꾸려고 `.T`를 사용함.

먼저 원소 수를 확인하고, 그 다음 새 axis에 이름을 붙이며, 마지막으로 reshape와 transpose를 구분함. Day 01에서는 transpose의 세부 API를 더 늘리지 않고 이 알아두기 경계까지만 다룸.


In [111]:
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])
target_column = target_flat.unsqueeze(1)
target_restored = target_column.squeeze(1)

print(target_flat)
print("flat:", tuple(target_flat.shape))
print("unsqueeze(1):", tuple(target_column.shape))
print("squeeze(1):", tuple(target_restored.shape))
print("squeeze(1):", target_restored.shape)



tensor([3., 5., 7., 9.])
flat: (4,)
unsqueeze(1): (4, 1)
squeeze(1): (4,)
squeeze(1): torch.Size([4])


### `squeeze`와 `unsqueeze`

- `unsqueeze(dim)`: 길이 1인 새 axis를 추가함
- `squeeze(dim)`: 지정한 길이 1 axis를 제거함
- 단순히 괄호를 붙였다 떼는 것이 아니라, 이후 연산에서 이 축의 의미가 무엇인지 확인함


In [ ]:
morning = torch.tensor([[18.0, 45.0], [21.0, 55.0]])
afternoon = torch.tensor([[25.0, 60.0], [20.0, 50.0]])

joined_samples = torch.cat([morning, afternoon], dim=0)
separate_groups = torch.stack([morning, afternoon], dim=0)

print("cat:", tuple(joined_samples.shape))
print("stack:", tuple(separate_groups.shape))


### `cat`과 `stack`

- `cat`: 기존 axis를 따라 이어 붙임
- `stack`: 새 axis를 만들어 여러 Tensor를 구분함
- 둘 다 실행된다는 이유만으로 같은 의미가 아님. 새 axis가 필요한지 먼저 결정함


### cat/stack 새 상황 적용

두 센서가 각각 `시간 2개 × 특성 2개`를 기록했다고 가정함. 두 기록을 하나의 긴 시간 흐름으로 이어 붙일지, `센서`라는 새 축을 남길지에 따라 연산이 달라짐. 실행 전에 각 결과 shape와 새 axis의 이름을 예상해봄.


In [35]:
sensor_a = torch.tensor([[10.0, 0.1], [11.0, 0.2]])
sensor_b = torch.tensor([[20.0, 0.3], [21.0, 0.4]])

print(sensor_a.shape)
print(sensor_b.shape)

long_timeline = torch.cat([sensor_a, sensor_b], dim=0)
separate_sensors = torch.stack([sensor_a, sensor_b], dim=0)

print("기존 시간 축을 늘린 결과:", tuple(long_timeline.shape))
print(long_timeline)
print(long_timeline.ndim)
print("새 센서 축을 만든 결과:", tuple(separate_sensors.shape))
print(separate_sensors)
print(separate_sensors.ndim)


torch.Size([2, 2])
torch.Size([2, 2])
기존 시간 축을 늘린 결과: (4, 2)
tensor([[10.0000,  0.1000],
        [11.0000,  0.2000],
        [20.0000,  0.3000],
        [21.0000,  0.4000]])
2
새 센서 축을 만든 결과: (2, 2, 2)
tensor([[[10.0000,  0.1000],
         [11.0000,  0.2000]],

        [[20.0000,  0.3000],
         [21.0000,  0.4000]]])
3


두 결과 중 하나를 고르는 질문은 “어느 함수가 더 좋은가?”가 아님.

1. 기존 axis 중 어느 길이를 늘리려는가?
2. 서로 다른 묶음을 구분할 새 axis가 필요한가?
3. 다음 계산은 그 새 axis를 이해하는가?

`cat` 결과와 `stack` 결과를 각각 `시간/특성`, `센서/시간/특성` 문장으로 설명해봄. 새 axis의 이름을 말할 수 없다면 `stack`이 필요한지 다시 확인함.


### 직접 확인하기 · 모델 입력 shape 만들기

`12`개 값을 `샘플 3개 × 특성 4개`로 바꾸는 한 줄을 작성함. `numel()`이 유지되는지 확인함.

<details><summary>Hint</summary>
첫 axis를 3으로 고정하고 둘째 axis는 직접 쓰거나 <code>-1</code>로 계산함.
</details>


In [36]:
practice_values = torch.arange(12)
# 아래 줄을 수정함
practice_matrix = practice_values
print("shape:", tuple(practice_matrix.shape), "원소 수:", practice_matrix.numel())


shape: (12,) 원소 수: 12


### NumPy와 최소 연결

이미 NumPy 배열이 있다면 Tensor로 가져올 수 있고, CPU Tensor를 다시 NumPy 배열로 볼 수도 있음. 오늘은 흐름을 끊지 않도록 변환을 알아보는 데서 멈춤.


In [37]:
import numpy as np

numpy_values = np.array([1.5, 2.5, 3.5], dtype=np.float32)
torch_values = torch.from_numpy(numpy_values)
back_to_numpy = torch_values.numpy()

print(type(numpy_values).__name__, numpy_values.dtype)
print(type(torch_values).__name__, torch_values.dtype)
print(type(back_to_numpy).__name__, back_to_numpy.dtype)


ndarray float32
Tensor torch.float32
ndarray float32


## 4. 실행됐는데 계산은 틀릴 수 있을까?

예측은 `[n, 1]`, 정답은 `[n]`인 상황을 확인함. 숫자 개수는 둘 다 `n`개라서 괜찮아 보이지만, PyTorch는 두 shape를 1:1로 맞추지 않을 수 있음.


In [71]:
prediction_column = torch.tensor([[2.0], [4.0], [6.0], [8.0]])
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])

print(prediction_column.shape)
print(prediction_column)
print(target_flat.shape)
print(target_flat)

silent_difference = prediction_column - target_flat
print("prediction shape:", tuple(prediction_column.shape))
print("target shape:", tuple(target_flat.shape))
print("difference shape:", tuple(silent_difference.shape))
print(silent_difference)


torch.Size([4, 1])
tensor([[2.],
        [4.],
        [6.],
        [8.]])
torch.Size([4])
tensor([3., 5., 7., 9.])
prediction shape: (4, 1)
target shape: (4,)
difference shape: (4, 4)
tensor([[-1., -3., -5., -7.],
        [ 1., -1., -3., -5.],
        [ 3.,  1., -1., -3.],
        [ 5.,  3.,  1., -1.]])


![broadcasting silent failure](assets/04_broadcasting_trap.svg)

### 핵심 문장

**실행됨 ≠ 의도한 계산임**

오류가 없다는 것은 PyTorch 규칙상 계산 가능했다는 뜻임. 각 예측과 같은 위치의 정답만 비교했다는 뜻은 아님.


In [58]:
print(target_flat.shape)
target_column = target_flat.unsqueeze(1)
safe_difference = prediction_column - target_column
print(f'unsqueeze {target_column.shape}')

print("맞춘 target shape:", tuple(target_column.shape))
print("1:1 difference shape:", tuple(safe_difference.shape))
print(safe_difference)
#
# assert prediction_column.shape == target_column.shape


torch.Size([4])
unsqueeze torch.Size([4, 1])
맞춘 target shape: (4, 1)
1:1 difference shape: (4, 1)
tensor([[-1.],
        [-1.],
        [-1.],
        [-1.]])


### 의도와 맞는 broadcasting, 침묵 실패 broadcasting

Broadcasting은 피해야 할 오류 이름이 아님. 모든 값에 같은 scalar 보정값을 더하려는 계산에서는 확장 규칙과 의도가 일치함. 반면 `[n,1]` prediction에서 `[n]` target을 빼면 샘플별 1:1 비교가 아니라 모든 조합 비교가 생김. 실행 전에 두 계산의 결과 shape를 예상함.


In [65]:
m1 = torch.FloatTensor([[3, 3]])
m2 = torch.FloatTensor([[2, 2]])

print(m1)
print(m1.shape)
print(m2)
print(m2.shape)
print(m1 + m2)

tensor([[3., 3.]])
torch.Size([1, 2])
tensor([[2., 2.]])
torch.Size([1, 2])
tensor([[5., 5.]])


In [70]:
m1 = torch.FloatTensor([[1, 2]])
m2 = torch.FloatTensor([[3], [4]])

print(m1.shape)
print(m1)
print(m2.shape)
print(m2)
m3 = m1 + m2
print(m3.shape)
print(m1 + m2)

torch.Size([1, 2])
tensor([[1., 2.]])
torch.Size([2, 1])
tensor([[3.],
        [4.]])
torch.Size([2, 2])
tensor([[4., 5.],
        [5., 6.]])


In [78]:
base_matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
scalar_bias = torch.tensor(0.5)

print(base_matrix.shape)
print(base_matrix)
print(scalar_bias.shape)
print(scalar_bias)

good_broadcast = base_matrix + scalar_bias
print("matrix + scalar shape:", tuple(good_broadcast.shape))
print(good_broadcast)
#
# bad_pairing = prediction_column - target_flat
#
# print("matrix + scalar shape:", tuple(good_broadcast.shape))
# print(good_broadcast)
# print("[n,1] - [n] shape:", tuple(bad_pairing.shape))
# print(bad_pairing)
#

torch.Size([2, 2])
tensor([[1., 2.],
        [3., 4.]])
torch.Size([])
tensor(0.5000)
matrix + scalar shape: (2, 2)
tensor([[1.5000, 2.5000],
        [3.5000, 4.5000]])


### Loss 직전 shape 확인

아래 실행에서는 잘못된 비교와 올바른 1:1 비교가 모두 마지막에 scalar MSE 하나를 만듦. 실행 전에 “scalar가 나왔으니 둘 다 올바르다”라고 결론 내릴 수 있는지 판단해봄. 실행 후에는 loss 숫자보다 `비교 직전 shape`와 `차이 Tensor의 shape`를 근거로 설명함.


In [87]:

print(f'prediction_column >> {prediction_column.shape}')
print(f'target_flat >> {target_flat.shape}')

wrong_difference = prediction_column - target_flat
right_difference = prediction_column - target_flat.unsqueeze(1)
a = target_flat.unsqueeze(1)
print(a.shape)
print(f'babo >>> {wrong_difference}')
print(f'hoho >>> {right_difference}')
wrong_scalar_loss = (wrong_difference ** 2).mean()
right_scalar_loss = (right_difference ** 2).mean()

print(wrong_scalar_loss)
print(right_scalar_loss)
#
# print("잘못된 경로: difference", tuple(wrong_difference.shape), "→ loss", tuple(wrong_scalar_loss.shape), wrong_scalar_loss.item())
# print("1:1 경로:    difference", tuple(right_difference.shape), "→ loss", tuple(right_scalar_loss.shape), right_scalar_loss.item())


prediction_column >> torch.Size([4, 1])
target_flat >> torch.Size([4])
torch.Size([4, 1])
babo >>> tensor([[-1., -3., -5., -7.],
        [ 1., -1., -3., -5.],
        [ 3.,  1., -1., -3.],
        [ 5.,  3.,  1., -1.]])
hoho >>> tensor([[-1.],
        [-1.],
        [-1.],
        [-1.]])
tensor(11.)
tensor(1.)


두 loss가 모두 scalar라는 사실은 `mean()`이 여러 값을 하나로 줄였다는 뜻일 뿐, 앞선 짝짓기가 옳았다는 보증이 아님. 다음 세 줄을 자신의 진단 순서로 적어봄.

> prediction shape 확인 → target shape와 axis 의미 확인 → difference shape가 의도한 짝인지 확인

모든 broadcasting을 금지하지 않음. 샘플별 정답 짝이 중요한 loss 직전에는 exact shape와 axis 의미를 적극적으로 확인함.


### loss-safe shape 습관

손실을 계산하기 전에 다음 두 질문을 먼저 확인함.

1. prediction과 target의 shape가 같은가?
2. 각 axis가 같은 의미인가?

단순히 원소 수가 같은지만 확인하면 침묵 실패를 놓칠 수 있음.


### 짧은 실습 · shape guard 통과시키기

아래 `target_attempt`을 수정해 prediction과 같은 shape로 만듦. 실행 결과가 `안전한 1:1 비교 준비됨`이 되면 완료함.

<details><summary>Hint</summary>
길이 1인 둘째 axis를 추가하는 연산을 사용함.
</details>


In [89]:
target_attempt = target_flat.unsqueeze(1) # 수정 대상

if target_attempt.shape == prediction_column.shape:
    print("안전한 1:1 비교 준비됨")
else:
    print("shape를 다시 확인함:", tuple(prediction_column.shape), tuple(target_attempt.shape))


안전한 1:1 비교 준비됨


In [90]:
features = torch.tensor([2.0, 3.0])
weights = torch.tensor([4.0, 5.0])

elementwise = features * weights
combined = features @ weights

print("* 결과:", elementwise, "shape:", tuple(elementwise.shape))
print("@ 결과:", combined, "shape:", tuple(combined.shape))


* 결과: tensor([ 8., 15.]) shape: (2,)
@ 결과: tensor(23.) shape: ()


### 관찰

- `*`: `[2×4, 3×5]`처럼 위치별 곱을 남김
- `@`: `2×4 + 3×5`처럼 입력 특성의 기여를 합쳐 하나의 값을 만듦
- 신경망에서는 여러 입력 특성과 weight를 결합해야 하므로 행렬곱이 필요함


![행렬곱 shape 확인 기준](assets/05_matmul_shape_guide.svg)


In [91]:
X = torch.tensor([[1.0, 2.0],
                  [2.0, 1.0],
                  [3.0, 2.0]])          # [batch=3, in=2]
W = torch.tensor([[2.0],
                  [1.0]])               # [in=2, out=1]

batch_output = X @ W
same_with_matmul = torch.matmul(X, W)

print("X:", tuple(X.shape), "W:", tuple(W.shape))
print("X @ W:", tuple(batch_output.shape))
print(batch_output)
print("matmul과 동일:", torch.equal(batch_output, same_with_matmul))


X: (3, 2) W: (2, 1)
X @ W: (3, 1)
tensor([[4.],
        [5.],
        [8.]])
matmul과 동일: True


### `X @ W` 첫 세 샘플 손계산 확인

코드를 실행하기 전에 첫 행 `[1, 2]`와 weight 열 `[2, 1]`을 손으로 계산함. 대응 위치를 곱한 뒤 더하는 순서를 적고, 둘째·셋째 행도 같은 규칙으로 계산해봄. 먼저 `(3,2) @ (2,1) → (3,1)`을 쓰고 값 계산으로 내려감.

1. 첫 샘플: `1×2 + 2×1 = ___`
2. 둘째 샘플: `2×2 + 1×1 = ___`
3. 셋째 샘플: `3×2 + 2×1 = ___`


In [92]:
contributions = X.unsqueeze(2) * W.unsqueeze(0)
print("샘플별 특성 기여:\n", contributions.squeeze(2))
print("기여를 더한 결과:\n", contributions.sum(dim=1))
print("X @ W 결과:\n", X @ W)


샘플별 특성 기여:
 tensor([[2., 2.],
        [4., 1.],
        [6., 2.]])
기여를 더한 결과:
 tensor([[4.],
        [5.],
        [8.]])
X @ W 결과:
 tensor([[4.],
        [5.],
        [8.]])


실행 결과에서 각 행의 두 기여가 하나의 출력으로 합쳐졌는지 확인함. 출력 열이 하나이므로 샘플마다 결과가 하나임. 이 장면의 목적은 행렬곱 공식을 암기하는 것이 아니라 `각 특성의 기여를 합쳐 샘플별 출력을 만든다`는 의미를 값과 shape에 함께 연결하는 것임.


### shape 계약 읽기

`[batch, in] @ [in, out] → [batch, out]`

- 맞닿는 `in` 축의 길이가 같아야 함
- `batch`와 `out` 축이 결과에 남음
- 결과 shape를 먼저 예상하면 긴 모델 코드에서도 오류 위치를 좁힐 수 있음


### 직접 확인하기 · 결과 shape 먼저 말하기

`[5, 3] @ [3, 2]`의 결과 shape를 실행 전에 말해봄. 어떤 axis가 맞닿고 어떤 axis가 남는지도 설명함.


In [ ]:
left = torch.ones(5, 3)
right = torch.ones(3, 2)
result = left @ right
print("실제 결과 shape:", tuple(result.shape))


### 행렬곱 새 상황 적용 — 남는 axis를 먼저 찾음

다음 세 건의 결과를 실행 전에 적어봄. `맞닿는 in 축`과 `결과에 남는 batch/out 축`을 말하는 것이 목표임.

1. 이미지 4장의 특성 6개를 출력 2개로 바꿈: `(4,6) @ (6,2) → ___`
2. 샘플 1개의 특성 5개를 출력 3개로 바꿈: `(1,5) @ (5,3) → ___`
3. `(7,4) @ (3,2)`는 실행 가능한가? 아니라면 어느 축 계약이 깨지는가?


In [93]:
matmul_cases = [
    (torch.ones(4, 6), torch.ones(6, 2)),
    (torch.ones(1, 5), torch.ones(5, 3)),
    (torch.ones(7, 4), torch.ones(3, 2)),
]

for index, (left_case, right_case) in enumerate(matmul_cases, start=1):
    try:
        output_case = left_case @ right_case
        print(f"case {index}: {tuple(left_case.shape)} @ {tuple(right_case.shape)} → {tuple(output_case.shape)}")
    except RuntimeError:
        print(f"case {index}: {tuple(left_case.shape)} @ {tuple(right_case.shape)} → 안쪽 축이 맞지 않아 실행할 수 없음")


case 1: (4, 6) @ (6, 2) → (4, 2)
case 2: (1, 5) @ (5, 3) → (1, 3)
case 3: (7, 4) @ (3, 2) → 안쪽 축이 맞지 않아 실행할 수 없음


오류가 난 세 번째 사례에서는 임의로 reshape해 숫자를 맞추지 않음. 왼쪽의 특성 수 4와 오른쪽 weight가 기대하는 입력 수 3 중 어떤 계약이 실제 데이터에 맞는지 먼저 확인해야 함. shape 수정은 의미 계약을 확인한 뒤 수행함.


### 낯선 예측 코드에서 두 문제 찾기

아래 코드는 샘플 3개, 특성 2개로 출력 하나를 만들고 target과 1:1로 비교하려는 코드임. 두 구간은 서로 독립적으로 실행되며, 하나는 오류를 내고 다른 하나는 오류 없이 이상한 shape를 만듦.

실행 전에 적어봄.

1. 행렬곱에서 서로 맞닿아야 하는 축은 무엇인가?
2. prediction `(3, 1)`과 target `(3,)`을 빼면 어떤 shape가 될 것 같은가?
3. 오류가 없다는 사실만으로 샘플별 1:1 비교라고 말할 수 있는가?


In [94]:
lab_features = torch.tensor([[1.0, 2.0],
                             [2.0, 1.0],
                             [3.0, 2.0]])
lab_target = torch.tensor([4.0, 5.0, 8.0])

# 문제 A: 행렬곱 입력 특성 수가 맞지 않음
weight_set_a = torch.ones(3, 1)
try:
    output_a = lab_features @ weight_set_a
    print("문제 A output shape:", tuple(output_a.shape))
except RuntimeError as error:
    print("문제 A:", type(error).__name__, "— 행렬곱 shape를 다시 확인함")

# 문제 B: 행렬곱은 되지만 target과의 짝이 의도와 다름
weight_set_b = torch.tensor([[2.0], [1.0]])
output_b = lab_features @ weight_set_b
difference_b = output_b - lab_target
print("문제 B prediction shape:", tuple(output_b.shape))
print("문제 B target shape:", tuple(lab_target.shape))
print("문제 B difference shape:", tuple(difference_b.shape))


문제 A: RuntimeError — 행렬곱 shape를 다시 확인함
문제 B prediction shape: (3, 1)
문제 B target shape: (3,)
문제 B difference shape: (3, 3)


### 원인과 최소 수정 기록

`오류/이상 출력 → 예상과 비교 → 원인 → 최소 수정 → 다시 실행` 순서로 기록함.

- 문제 A의 관찰: ___
- 문제 A의 원인: 왼쪽 특성 축 길이 ___와 오른쪽 입력 축 길이 ___가 다름
- 문제 A의 최소 수정: 데이터 의미를 확인한 뒤 weight의 입력 축을 ___에 맞춤
- 문제 B의 관찰: difference shape가 ___임
- 문제 B의 원인: prediction의 출력 축과 target의 축 표현이 ___
- 문제 B의 최소 수정: target에 길이 1인 ___ 축을 추가함

아래 셀의 두 `*_attempt` 줄만 수정함. 전체 코드를 새로 쓰지 않음.

<details><summary>막힐 때 확인할 점</summary>
행렬곱은 `[batch, in] @ [in, out]`을 먼저 읽고, loss 전에는 prediction과 target의 shape와 axis 의미를 함께 확인함.
</details>


In [95]:
weight_attempt = torch.ones(3, 1)   # 원인을 확인한 뒤 이 줄만 수정함
target_attempt = lab_target            # 원인을 확인한 뒤 이 줄만 수정함

inner_axes_match = lab_features.shape[1] == weight_attempt.shape[0]
if not inner_axes_match:
    print("다시 실행 전 확인: 행렬곱 안쪽 축이 아직 맞지 않음")
else:
    retry_prediction = lab_features @ weight_attempt
    if retry_prediction.shape != target_attempt.shape:
        print("다시 실행 전 확인: prediction과 target shape가 아직 다름")
    else:
        retry_difference = retry_prediction - target_attempt
        retry_loss = (retry_difference ** 2).mean()
        print("다시 실행 성공")
        print("prediction shape:", tuple(retry_prediction.shape))
        print("difference shape:", tuple(retry_difference.shape))
        print("loss:", retry_loss.item())


다시 실행 전 확인: 행렬곱 안쪽 축이 아직 맞지 않음


In [96]:
weight_attempt = torch.ones(3, 1)   # 원인을 확인한 뒤 이 줄만 수정함
target_attempt = lab_target

# 안됨
print(weight_attempt.shape)
print(target_attempt.shape)

torch.Size([3, 1])
torch.Size([3])


다시 실행이 성공하면 두 수정이 각각 어떤 문제를 해결했는지 분리해 설명함. 행렬곱 수정은 입력 특성 축을 맞추고, target 수정은 샘플별 출력 축을 맞춤. 단순히 오류가 사라졌다는 말에서 멈추지 않고 결과가 `(샘플 3개, 출력 1개)`의 의도와 맞는지 확인함.


## 6. 가장 작은 모델은 어떻게 prediction을 만들까?

![smallest linear model](assets/06_linear_model.svg)

`ŷ = wx + b`에서 `x`는 입력, `ŷ`는 예측, `w`와 `b`는 모델 안에서 바꿀 값임.


In [102]:
x_train = torch.tensor([1.0, 2.0])
target_train = torch.tensor([3.0, 5.0])

w = torch.tensor(0.0)
b = torch.tensor(0.0)
prediction = w * x_train + b

print("입력:", x_train)
print("현재 w, b:", w.item(), b.item())
print("현재 prediction:", prediction)
print("target:", target_train)


입력: tensor([1., 2.])
현재 w, b: 0.0 0.0
현재 prediction: tensor([0., 0.])
target: tensor([3., 5.])


### 같은 x에서 w 변화와 b 변화를 분리함

`ŷ = wx+b`에서 두 parameter를 동시에 바꾸면 역할이 섞여 보임. 같은 입력 `[1, 2, 4]`에서 w만 1 올린 경우와 b만 1 올린 경우를 비교함. 실행 전에 각 prediction의 증가량이 모든 위치에서 같은지 예상해봄.


In [105]:
x_sensitivity = torch.tensor([1.0, 2.0, 4.0])
base_w, base_b = 2.0, 0.5

base_prediction = base_w * x_sensitivity + base_b
w_changed_prediction = (base_w + 1.0) * x_sensitivity + base_b
b_changed_prediction = base_w * x_sensitivity + (base_b + 1.0)

print(f'x >>> {x_sensitivity}')
print(w_changed_prediction)
print(b_changed_prediction )
# print("x:", x_sensitivity)
# print("기준 prediction:", base_prediction)
# print("w만 +1 했을 때 변화:", w_changed_prediction - base_prediction)
# print("b만 +1 했을 때 변화:", b_changed_prediction - base_prediction)


x >>> tensor([1., 2., 4.])
tensor([ 3.5000,  6.5000, 12.5000])
tensor([3.5000, 5.5000, 9.5000])


관찰한 변화량을 이용해 설명해봄.

- w 변화는 입력 크기와 어떤 관계가 있는가?
- b 변화는 서로 다른 x에서 어떻게 나타나는가?
- 두 parameter의 gradient가 서로 다른 숫자로 나오는 것이 자연스러운 이유는 무엇인가?

한 번의 예에서 보인 방향을 모든 상태에 일반화하지 않음. 이 장면은 두 parameter가 prediction에 서로 다른 방식으로 기여한다는 사실만 확인함.


### 처음부터 잘 맞힐 필요는 없음

지금 예측은 틀림. 그러나 모델의 학습 가능성은 첫 예측의 정확함이 아니라 다음 구조에 있음.

- 바꿀 내부 값 `w`, `b`가 있음
- prediction과 target의 차이를 숫자로 만들 수 있음
- 그 차이를 이용해 내부 값을 바꿀 수 있음


## 7. 틀린 정도를 숫자 하나로 어떻게 요약할까?

두 prediction이 각각 얼마나 틀렸는지 계산하고, 제곱하고, 평균냄. 이 계산에 이름을 붙이면 **Mean Squared Error, 평균제곱오차(MSE)**임.


![MSE flow](assets/07_mse_flow.svg)


In [107]:
difference = prediction - target_train
squared_error = difference ** 2
mse = squared_error.mean()

print("차이:", difference)
print("차이의 제곱:", squared_error)
print("MSE:", mse.item())


차이: tensor([-3., -5.])
차이의 제곱: tensor([ 9., 25.])
MSE: 17.0


### 수식 읽기

`MSE = mean((prediction - target) ** 2)`

- 차이: 각 예측이 어느 방향으로 얼마나 벗어났는지
- 제곱: 음수와 양수가 상쇄되지 않게 하고 큰 오차에 더 큰 값을 줌
- 평균: 샘플 수가 달라도 한 개의 대표 손실로 비교하기 쉬워짐
- 제곱근은 취하지 않음. 제곱근까지 취하면 RMSE라는 다른 이름을 사용함


### 짧은 실습 · 두 prediction 비교

아래 두 prediction의 MSE를 실행 전에 비교해봄. 각 위치의 오차를 먼저 보면 근거를 만들 수 있음.


In [110]:
prediction_a = torch.tensor([2.0, 6.0])
prediction_b = torch.tensor([1.0, 5.0])

loss_a = ((prediction_a - target_train) ** 2).mean()
loss_b = ((prediction_b - target_train) ** 2).mean()

print("prediction A의 MSE:", loss_a.item())
print("prediction B의 MSE:", loss_b.item())


prediction A의 MSE: 1.0
prediction B의 MSE: 2.0


### 같은 MSE, 다른 샘플별 error pattern

MSE는 샘플별 제곱오차를 평균한 요약임. 따라서 같은 scalar MSE가 나와도 어느 샘플이 얼마나 틀렸는지는 다를 수 있음. 실행 전에 패턴 A와 B의 제곱오차 평균을 계산하고, 어느 쪽이 더 고르게 틀렸는지 예상해봄.


In [109]:
pattern_target = torch.zeros(4)
pattern_a = torch.tensor([-1.0, 1.0, -1.0, 1.0])
root_two = torch.sqrt(torch.tensor(2.0))
pattern_b = torch.tensor([0.0, 0.0, -root_two.item(), root_two.item()])

for label, pred in {"A": pattern_a, "B": pattern_b}.items():
    errors = pred - pattern_target
    squared = errors ** 2
    print(f"패턴 {label} | error={errors.tolist()} | squared={squared.tolist()} | MSE={squared.mean().item():.3f}")


패턴 A | error=[-1.0, 1.0, -1.0, 1.0] | squared=[1.0, 1.0, 1.0, 1.0] | MSE=1.000
패턴 B | error=[0.0, 0.0, -1.4142135381698608, 1.4142135381698608] | squared=[0.0, 0.0, 1.9999998807907104, 1.9999998807907104] | MSE=1.000


두 MSE가 같아도 패턴 A는 모든 샘플이 비슷하게 틀리고, 패턴 B는 일부 샘플이 정확한 대신 나머지 오차가 큼. 따라서 scalar loss 하나만으로 “모든 샘플이 비슷하게 좋아졌다”고 말할 수 없음.

MSE 감소는 현재 MSE 계약 아래 현재 데이터의 평균 제곱오차가 줄었다는 근거임. 보지 않은 데이터의 성능, 공정성, 인과 관계까지 증명하지 않음. 오늘은 error analysis 전체로 넓히지 않고 요약이 감추는 정보가 있다는 경계까지만 다룸.


### Broadcasting callback

MSE 코드가 짧아도 prediction과 target shape가 다르면 의도하지 않은 비교를 평균낼 수 있음. 손실 직전에 shape와 axis 의미를 다시 확인함.


## 8. 어느 방향으로 값을 바꿔야 할까?

손실 `17`은 현재 많이 틀렸다는 사실을 보여주지만, `w`와 `b`를 키울지 줄일지는 직접 말해주지 않음. 값이 조금 바뀔 때 손실이 어느 방향으로 얼마나 민감하게 변하는지 알아야 함.


### 한 parameter를 조금 움직여 loss 변화를 봄

먼저 `b=0`으로 고정하고 `w` 하나만 움직임. 현재 위치 `w=0`의 왼쪽 `-0.1`과 오른쪽 `+0.1`에서 loss를 비교함.

![현재 위치에서 loss 곡선의 국소 기울기 읽기](assets/loss_curve_local_slope.svg)

그림에서 곡선 전체의 최저점을 바로 찾는 것이 목표가 아님. 현재 점 주변에서 오른쪽으로 아주 조금 움직일 때 loss가 올라가는지 내려가는지, 변화가 얼마나 가파른지 읽는 것이 목표임.


In [ ]:
def loss_with_fixed_b(w_value):
    local_prediction = w_value * x_train  # b는 0으로 고정함
    return ((local_prediction - target_train) ** 2).mean().item()

for nearby_w in [-0.1, 0.0, 0.1]:
    print(f"w={nearby_w:+.1f} → loss={loss_with_fixed_b(nearby_w):.3f}")

local_change_rate = (loss_with_fixed_b(0.1) - loss_with_fixed_b(-0.1)) / 0.2
print(f"양옆 두 점으로 본 현재 근처 변화율: {local_change_rate:.3f}")


### 그림과 수치를 연결함

- `w=-0.1 → 0 → +0.1`로 움직일 때 loss가 감소함
- 현재 위치에서는 w를 조금 키우는 쪽이 loss를 낮추는 방향임
- 변화율의 음수 부호는 오른쪽으로 갈수록 loss가 내려가는 방향을 나타냄
- 절댓값이 크다는 것은 현재 근처에서 변화가 가파르다는 뜻임

한 변수의 현재 위치 변화율을 **derivative, 도함수 값**으로 표현함. 바꿀 parameter가 `w`, `b`처럼 여러 개라면 각 parameter에 대한 현재 위치의 derivative를 모은 정보가 **gradient**임.

Gradient는 목적지 좌표가 아니라 **현재 위치의 국소 변화 정보**임. 한 번 움직인 뒤에는 새 위치에서 다시 계산해야 함.


### 왜 자동 계산이 필요한가?

지금은 `w` 하나와 두 데이터라 양옆의 loss를 직접 비교할 수 있었음. 실제 모델은 parameter가 매우 많고 계산 경로도 길어짐. 모든 parameter의 국소 변화 정보를 매번 손으로 전개하기 어렵기 때문에, PyTorch가 forward 계산 연결을 기록하고 gradient를 계산하게 함. 이 기능이 **Autograd**임.

다음 셀에서는 `requires_grad=True`로 추적할 parameter를 표시하고, `backward()`로 gradient를 계산함. 아직 parameter update는 하지 않음.


![computational graph forward and backward](assets/08_computational_graph.svg)


In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = w * x_train + b
loss = ((prediction - target_train) ** 2).mean()

print("prediction:", prediction)
print("loss:", loss.item())
print("prediction의 grad_fn:", type(prediction.grad_fn).__name__)
print("loss의 grad_fn:", type(loss.grad_fn).__name__)


### `requires_grad=True`

`w`와 `b`까지 이어지는 계산을 추적해 나중에 손실의 기울기를 구할 수 있게 함. `x_train`과 `target_train`은 오늘 바꿀 parameter가 아니므로 추적 대상으로 만들지 않음.

실행 전 예상: `loss.backward()`를 호출하면 `w`와 `b` 자체가 바뀔까, 아니면 다른 곳에 정보가 저장될까?


In [ ]:
w_before_backward = w.detach().clone()
b_before_backward = b.detach().clone()

loss.backward()

print("backward 전후 w 동일:", torch.equal(w_before_backward, w.detach()))
print("backward 전후 b 동일:", torch.equal(b_before_backward, b.detach()))
print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())


### 계산 그래프의 사용 흐름

forward를 실행할 때 현재 연산 연결로 computational graph가 만들어지고, `backward()`는 loss에서 시작해 그 연결을 거꾸로 따라감. 첫 backward가 끝난 같은 loss를 그대로 다시 사용하면 이미 해제된 연결을 다시 따라가려는 오류가 날 수 있음.

아래 셀은 의도된 오류를 `try/except` 안에서만 확인하므로 Run All을 멈추지 않음. 실행 전에 첫 backward와 둘째 backward 중 어디에서 문제가 생길지 예상해봄.


In [ ]:
graph_w = torch.tensor(0.0, requires_grad=True)
graph_b = torch.tensor(0.0, requires_grad=True)
graph_prediction = graph_w * x_train + graph_b
graph_loss = ((graph_prediction - target_train) ** 2).mean()

graph_loss.backward()
print("첫 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())

try:
    graph_loss.backward()
except RuntimeError as error:
    print("같은 loss의 두 번째 backward가 중단됨:", type(error).__name__)
    print("이유: 첫 backward 뒤 같은 graph를 그대로 재사용하려 했음")

graph_w.grad.zero_()
graph_b.grad.zero_()
fresh_graph_prediction = graph_w * x_train + graph_b
fresh_graph_loss = ((fresh_graph_prediction - target_train) ** 2).mean()
fresh_graph_loss.backward()
print("fresh forward 뒤 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())


이 장면에서 얻을 습관은 오류를 피하려고 옵션을 외우는 것이 아님. 매 학습 단계에서 현재 parameter로 fresh prediction과 fresh loss를 만들고 그 loss에 backward를 수행함. `retain_graph` 사용법 목록은 Day 01 범위에 넣지 않음.

오류가 보이면 PyTorch 설치 문제라고 단정하지 않고 “같은 loss 객체에 backward를 반복했는가?”를 먼저 확인함.


### gradient의 부호와 크기를 현재 위치의 정보로 읽음

현재 `w.grad=-13`, `b.grad=-8`을 보고 두 문장을 구분함.

1. 음수 부호는 현재 위치에서 해당 parameter를 증가시키면 loss가 감소하는 방향임을 시사함.
2. 절댓값 13과 8은 현재 단위와 입력 scale 아래의 민감도임.

실행 전에 w를 아주 조금 늘렸을 때와 줄였을 때 어느 쪽 loss가 작을지 예상해봄.


In [ ]:
def loss_at(w_value, b_value):
    local_prediction = w_value * x_train + b_value
    return ((local_prediction - target_train) ** 2).mean().item()

epsilon = 0.01
print("현재 loss:", loss_at(0.0, 0.0))
print("w를 +0.01:", loss_at(epsilon, 0.0))
print("w를 -0.01:", loss_at(-epsilon, 0.0))
print("b를 +0.01:", loss_at(0.0, epsilon))
print("b를 -0.01:", loss_at(0.0, -epsilon))


`|w.grad| > |b.grad|`라는 한 줄만 보고 “w가 더 중요한 parameter”라고 결론 내리지 않음. 입력의 크기, parameter 단위, 현재 위치가 다르면 gradient scale도 달라질 수 있음. Gradient는 목적지 좌표가 아니라 현재 위치의 국소 방향 정보이며, update 뒤에는 새 위치에서 다시 계산해야 함.


### evidence로 구분함

- `backward 전후 ... 동일: True`: parameter 값은 바뀌지 않았음
- `w.grad`, `b.grad`: 기울기는 `.grad`에 저장됐음
- `backward()`는 update가 아니라 gradient 계산임

기울기의 부호는 선악 표시가 아님. 현재 위치에서 해당 값을 증가시킬 때 손실이 어느 방향으로 변하는지 알려주는 신호임.


### 기울기가 기본적으로 누적되는지 확인함

같은 현재 parameter로 prediction과 loss를 새로 만들고, `.grad`를 비우지 않은 채 두 번째 `backward()`를 실행함.


In [ ]:
first_grad_w = w.grad.item()
first_grad_b = b.grad.item()

prediction_again = w * x_train + b
loss_again = ((prediction_again - target_train) ** 2).mean()
loss_again.backward()

print("첫 gradient:", first_grad_w, first_grad_b)
print("reset 없는 두 번째 backward 뒤:", w.grad.item(), b.grad.item())


### 기울기 누적 확인 — reset 누락 찾기

같은 현재 parameter에서 fresh loss를 두 번 만들더라도 `.grad`를 비우지 않으면 두 gradient가 누적됨. 아래 셀에서 첫 번째 실험은 reset을 생략하고, 두 번째 실험은 각 backward 뒤 reset함. 실행 전에 두 번째 기록이 각각 `-26`과 `-13` 중 무엇일지 예상해봄.


In [ ]:
def collect_grad_trace(reset_each_time):
    trace_w = torch.tensor(0.0, requires_grad=True)
    trace_b = torch.tensor(0.0, requires_grad=True)
    records = []
    for repeat in range(1, 3):
        trace_prediction = trace_w * x_train + trace_b
        trace_loss = ((trace_prediction - target_train) ** 2).mean()
        trace_loss.backward()
        records.append((repeat, trace_w.grad.item(), trace_b.grad.item()))
        if reset_each_time:
            trace_w.grad.zero_()
            trace_b.grad.zero_()
    return records

print("reset 누락:", collect_grad_trace(False))
print("매번 reset:", collect_grad_trace(True))


오류→예상→실행→원인→수정→retry 순서로 진단 기록을 남겨봄.

- 관찰한 이상: 둘째 `.grad`가 ___임.
- 원인 후보: 이전 단계의 ___가 남아 있음.
- 최소 수정: 다음 단계를 시작하기 전에 ___를 정리함.
- retry 근거: 같은 현재 상태의 각 backward가 다시 ___를 만듦.

누적 자체가 언제나 버그인 것은 아님. 오늘 계약은 한 학습 단계에 한 loss의 gradient만 사용하는 것이므로 reset 누락을 오류로 판정함.


### 관찰

두 번째 값은 첫 번째를 덮어쓴 것이 아니라 같은 기울기가 더해진 값임. 다음 학습 단계에서 현재 단계의 기울기만 쓰려면 직접 비워야 함.


In [ ]:
w.grad.zero_()
b.grad.zero_()
print("reset 뒤:", w.grad.item(), b.grad.item())
print("parameter는 여전히:", w.item(), b.item())


### 선택 학습 · 지금 나온 gradient를 손으로 읽기

두 샘플의 MSE를 쓰면 다음과 같음.

`L = ((w·1+b-3)² + (w·2+b-5)²) / 2`

현재 `w=0`, `b=0`에서 `w`의 기울기는 `-13`, `b`의 기울기는 `-8`임. 미분 증명 자체보다, 음수 gradient를 update 식에서 빼면 두 parameter가 증가한다는 연결을 읽음.


## 9. gradient를 사용해 실제 parameter를 바꿈

지금까지는 방향 정보를 계산했을 뿐 값은 그대로였음. 이제 별도의 update를 수행하고, 바뀐 값으로 prediction과 loss를 새로 계산함.


### 수동 update 손계산

앞에서 `backward()`로 확인한 값만 사용해 한 단계 update를 손으로 계산함. 실제 update 셀과 결과 그림은 이 시도 뒤에 실행함.

- 현재 `w=0`, `b=0`, `w.grad=-13`, `b.grad=-8`, 학습률 `0.1`임
- `new_w = 0 - 0.1 × ( ___ ) = ___`
- `new_b = 0 - 0.1 × ( ___ ) = ___`
- 첫 새 prediction: `new_w × 1 + new_b = ___`
- 둘째 새 prediction: `new_w × 2 + new_b = ___`
- 새 제곱오차 두 개와 평균을 계산하면 new loss는 ___임

개인 계산 3분 → 짝과 부호·평균 확인 2분 순서로 진행함. 아직 아래의 실제 update 셀을 실행하지 않음.


In [ ]:
manual_scaffold = {
    "new_w": None,
    "new_b": None,
    "prediction_1": None,
    "prediction_2": None,
    "new_loss": None,
}

completed = sum(value is not None for value in manual_scaffold.values())
print(f"손계산 기록: {completed}/{len(manual_scaffold)}개 입력함")
print(manual_scaffold)


계산이 막히면 완성 결과 대신 다음 순서만 확인함.

> update 식의 빼기 확인 → 음수 gradient의 괄호 확인 → 새 parameter로 prediction 다시 계산 → target과 차이 → 제곱 → 평균

짝과 비교할 때는 숫자만 맞추지 않고 부호와 평균 계산의 근거를 말함. 시도와 짝 확인이 끝난 뒤 다음 셀을 실행해 실제 update 결과와 대조함.


In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
step_size = 0.1

# 1) parameter before → prediction → loss
prediction_before = w * x_train + b
loss_before = ((prediction_before - target_train) ** 2).mean()

# 2) gradient 계산
loss_before.backward()
grad_w = w.grad.item()
grad_b = b.grad.item()

print(f"parameter before: w={w.item():.3f}, b={b.item():.3f}")
print("prediction before:", prediction_before.detach())
print(f"loss before: {loss_before.item():.3f}")
print(f"gradient: w.grad={grad_w:.3f}, b.grad={grad_b:.3f}")

# 3) 실제 parameter update
with torch.no_grad():
    w -= step_size * w.grad
    b -= step_size * b.grad

print(f"parameter after: w={w.item():.3f}, b={b.item():.3f}")

# 4) 다음 단계를 위한 gradient reset
w.grad.zero_()
b.grad.zero_()

# 5) 바뀐 parameter로 fresh prediction과 new loss
prediction_after = w * x_train + b
loss_after = ((prediction_after - target_train) ** 2).mean()

print("new prediction:", prediction_after.detach())
print(f"new loss: {loss_after.item():.3f}")
print(f"gradient after reset: {w.grad.item():.1f}, {b.grad.item():.1f}")


![manual parameter update before and after](assets/09_manual_update.svg)


### 한 번의 학습 단계에서 읽을 증거

1. parameter before
2. prediction before
3. loss before
4. gradient
5. parameter after
6. **바뀐 parameter로 다시 만든** new prediction
7. new loss

`backward()` 뒤가 아니라 `torch.no_grad()` 안의 뺄셈 뒤에 parameter가 바뀜.


### 왜 `torch.no_grad()`가 필요할까?

update 식 자체를 다음 gradient 계산을 위한 computational graph에 기록하지 않기 위해 사용함. gradient를 없애는 명령이 아니며, `backward()`를 대신하지도 않음. update가 끝난 뒤 `.grad.zero_()`로 누적 공간을 따로 정리함.


### no_grad, detach, item의 책임을 섞지 않음

`torch.no_grad()`는 블록 안의 새 연산을 graph에 기록하지 않게 함. `detach()`는 현재 graph 연결에서 떼어낸 Tensor 관점을 만들고, `item()`은 원소 하나짜리 Tensor 값을 Python 숫자로 꺼냄. 셋 모두 parameter update나 backward를 대신하지 않음.

아래 셀에서 각 결과의 Python/Tensor 여부와 `requires_grad`를 관찰함. 실행 전에 `item()` 결과에 `.backward()`를 호출할 수 있을지 예상해봄.


In [ ]:
demo_value = torch.tensor(3.0, requires_grad=True)
tracked_result = demo_value * 2
detached_result = tracked_result.detach()
python_number = tracked_result.item()
with torch.no_grad():
    untracked_result = demo_value * 2

print("tracked:", type(tracked_result).__name__, tracked_result.requires_grad)
print("detached:", type(detached_result).__name__, detached_result.requires_grad)
print("item:", type(python_number).__name__)
print("no_grad 안 결과:", type(untracked_result).__name__, untracked_result.requires_grad)


오늘 핵심는 update를 `no_grad` 안에서 수행하고, 다음 단계 전에 `.grad`를 따로 reset하는 것임. 출력 편의를 위한 `detach()`와 `item()`을 loss 계산 중간에 습관적으로 넣지 않음. 이 장면은 세 이름을 API 목록으로 확장하지 않고 책임을 혼동하지 않는 수준에서 닫음.


### 반복 가능한 수동 학습 구조

`prediction → loss → backward → manual update → grad reset → fresh prediction`

이 순서의 각 줄은 책임이 다름. 코드 순서를 외우기보다 `계산 → 방향 → 변경 → 정리 → 재확인`으로 읽음.


In [ ]:
w_loop = torch.tensor(0.0, requires_grad=True)
b_loop = torch.tensor(0.0, requires_grad=True)

for step in range(1, 5):
    pred_loop = w_loop * x_train + b_loop
    loss_loop = ((pred_loop - target_train) ** 2).mean()
    loss_loop.backward()

    with torch.no_grad():
        w_loop -= 0.1 * w_loop.grad
        b_loop -= 0.1 * b_loop.grad

    w_loop.grad.zero_()
    b_loop.grad.zero_()

    fresh_loss = ((w_loop * x_train + b_loop - target_train) ** 2).mean()
    print(f"step {step}: w={w_loop.item():.4f}, b={b_loop.item():.4f}, fresh loss={fresh_loss.item():.6f}")


### 다른 숫자에 같은 학습 구조 적용하기

새 데이터는 `x=[1,3]`, target은 `[2,6]`, 시작 parameter는 `w=0.5`, `b=0.5`임. 모델·MSE·학습 loop의 뼈대는 준비되어 있음.

학생이 맡을 일은 세 가지임.

1. 실행 전에 첫 prediction 두 개와 첫 loss를 직접 계산해 빈칸에 기록함
2. 학습률을 `0.03` 또는 `0.05` 중 하나로 선택하고 선택 이유를 한 문장으로 적음
3. 세 단계의 fresh loss 경로를 보고 처음 계산과 어떻게 달라졌는지 설명함

무작위로 코드를 새로 작성하는 실습이 아니라, 같은 구조를 새 숫자에서 읽고 선택 근거를 만드는 실습임.


In [ ]:
transfer_x = torch.tensor([1.0, 3.0])
transfer_target = torch.tensor([2.0, 6.0])
transfer_start_w = 0.5
transfer_start_b = 0.5

# 손계산 결과를 먼저 채움
first_prediction_attempt = [None, None]
first_loss_attempt = None

# 둘 중 하나를 선택함: 0.03 또는 0.05
step_size_choice = None
transfer_steps = 3

manual_ready = all(value is not None for value in first_prediction_attempt) and first_loss_attempt is not None
choice_ready = step_size_choice in {0.03, 0.05}

if not manual_ready:
    print("먼저 첫 prediction 두 개와 첫 loss를 직접 계산해 기록함")
elif not choice_ready:
    print("학습률을 0.03 또는 0.05 중 하나로 선택함")
else:
    actual_first_prediction = transfer_start_w * transfer_x + transfer_start_b
    actual_first_loss = ((actual_first_prediction - transfer_target) ** 2).mean()
    print("내가 기록한 첫 prediction:", first_prediction_attempt)
    print("실제 첫 prediction:", actual_first_prediction.tolist())
    print("내가 기록한 첫 loss:", first_loss_attempt)
    print("실제 첫 loss:", actual_first_loss.item())

    transfer_w = torch.tensor(transfer_start_w, requires_grad=True)
    transfer_b = torch.tensor(transfer_start_b, requires_grad=True)
    for step in range(1, transfer_steps + 1):
        transfer_prediction = transfer_w * transfer_x + transfer_b
        transfer_loss = ((transfer_prediction - transfer_target) ** 2).mean()
        transfer_loss.backward()
        with torch.no_grad():
            transfer_w -= step_size_choice * transfer_w.grad
            transfer_b -= step_size_choice * transfer_b.grad
        transfer_w.grad.zero_()
        transfer_b.grad.zero_()
        transfer_fresh_prediction = transfer_w * transfer_x + transfer_b
        transfer_fresh_loss = ((transfer_fresh_prediction - transfer_target) ** 2).mean()
        print(
            f"step {step}: w={transfer_w.item():.4f}, b={transfer_b.item():.4f}, "
            f"prediction={[round(v, 4) for v in transfer_fresh_prediction.tolist()]}, "
            f"fresh loss={transfer_fresh_loss.item():.6f}"
        )


실행 뒤에는 마지막 loss 하나만 보고 끝내지 않음.

- 손으로 계산한 첫 prediction/loss와 실행 결과가 맞는가?
- 선택한 학습률에서 세 fresh loss는 어떤 경로를 보였는가?
- w와 b는 어느 방향으로 움직였는가?
- 새 prediction은 target에 어떤 방식으로 가까워졌는가?
- 다른 학습률을 고른 짝과 비교하면 이동 크기와 loss 경로가 어떻게 다른가?

이 작은 학습 데이터의 loss 감소만으로 보지 않은 데이터 성능을 주장하지 않음.


### 수동 학습 loop 오류 찾기

아래 loop에는 세 가지 책임이 빠져 있음. update가 없고, gradient reset이 없고, 다음 단계의 fresh forward도 없음. 의도된 오류는 예외 안에서 제어되므로 전체 실행은 멈추지 않음.

실행 전에 다음을 예상함. 첫 backward 뒤 parameter가 바뀌는가? 둘째 반복에서 같은 loss를 다시 backward할 수 있는가? `.grad`를 비우지 않은 사실은 어디에서 확인할 수 있는가?


In [ ]:
broken_w = torch.tensor(0.0, requires_grad=True)
broken_b = torch.tensor(0.0, requires_grad=True)
broken_prediction = broken_w * x_train + broken_b
broken_loss = ((broken_prediction - target_train) ** 2).mean()

for broken_step in range(1, 3):
    try:
        broken_loss.backward()
        print(f"broken step {broken_step}: w={broken_w.item():.1f}, b={broken_b.item():.1f}, grad=({broken_w.grad.item():.1f}, {broken_b.grad.item():.1f})")
    except RuntimeError as error:
        print(f"broken step {broken_step}: 같은 loss 재사용으로 {type(error).__name__} 발생")
        break


최소 수정안을 코드 줄이 아니라 책임 순서로 먼저 적어봄.

1. 매 단계 현재 parameter로 ___와 ___를 새로 만듦.
2. backward 뒤 `no_grad` 안에서 ___를 수행함.
3. 다음 단계 전에 ___를 정리함.
4. 수정 뒤 parameter 변화와 ___ loss를 다시 확인함.

아래 세 표식을 모두 `True`로 바꾸면 준비된 retry가 실행됨. 실행 뒤 broken 결과와 evidence를 비교함.


In [ ]:
repair_fresh_forward = False
repair_parameter_update = False
repair_gradient_reset = False

if not all([repair_fresh_forward, repair_parameter_update, repair_gradient_reset]):
    print("세 책임의 최소 수정 위치를 설명한 뒤 표식을 True로 바꿈")
else:
    fixed_w = torch.tensor(0.0, requires_grad=True)
    fixed_b = torch.tensor(0.0, requires_grad=True)
    for fixed_step in range(1, 3):
        fixed_prediction = fixed_w * x_train + fixed_b
        fixed_loss = ((fixed_prediction - target_train) ** 2).mean()
        fixed_loss.backward()
        with torch.no_grad():
            fixed_w -= 0.1 * fixed_w.grad
            fixed_b -= 0.1 * fixed_b.grad
        fixed_w.grad.zero_()
        fixed_b.grad.zero_()
        fixed_fresh_loss = ((fixed_w * x_train + fixed_b - target_train) ** 2).mean()
        print(f"fixed step {fixed_step}: w={fixed_w.item():.4f}, b={fixed_b.item():.4f}, fresh loss={fixed_fresh_loss.item():.6f}")


### 직접 확인하기 · 학습 한 단계의 순서 복원

아래 다섯 역할을 실제 코드 순서로 적어봄.

- gradient reset
- parameter update
- prediction과 loss
- backward
- fresh prediction과 new loss

<details><summary>Hint</summary>
아직 없는 gradient를 사용할 수 없고, 이전 gradient를 다음 단계에 섞지 않아야 함.
</details>


In [ ]:
student_order = []  # 역할 이름을 순서대로 추가해봄
print("내가 정한 순서:", student_order if student_order else "아직 작성하지 않음")


## 10. 한 번에 움직이는 크기가 다르면 무엇이 달라질까?

같은 데이터, 같은 시작값, 같은 8단계를 사용하고 parameter를 움직이는 크기만 바꿈. 먼저 결과를 보고 그 크기의 이름을 붙임.


In [ ]:
def run_step_size_trial(step_size, steps=8):
    trial_w = torch.tensor(0.0, requires_grad=True)
    trial_b = torch.tensor(0.0, requires_grad=True)
    history = []

    for _ in range(steps):
        trial_prediction = trial_w * x_train + trial_b
        trial_loss = ((trial_prediction - target_train) ** 2).mean()
        history.append(trial_loss.item())
        trial_loss.backward()

        with torch.no_grad():
            trial_w -= step_size * trial_w.grad
            trial_b -= step_size * trial_b.grad

        trial_w.grad.zero_()
        trial_b.grad.zero_()

    final_prediction = trial_w * x_train + trial_b
    final_loss = ((final_prediction - target_train) ** 2).mean()
    return history, trial_w.item(), trial_b.item(), final_prediction.detach(), final_loss.item()


### 실행 전 관찰 질문

- 실험 A, B, C 중 손실이 거의 줄지 않는 경우는 무엇일까?
- 처음에는 줄더라도 정답을 지나쳐 흔들릴 수 있을까?
- 8단계 뒤 loss 하나만 보지 말고 중간 경로도 함께 확인함


In [ ]:
trials = {"실험 A": 0.001, "실험 B": 0.1, "실험 C": 0.5}

for label, size in trials.items():
    history, final_w, final_b, final_pred, final_loss = run_step_size_trial(size)
    compact_history = [round(value, 6) for value in history]
    print(f"{label} · 크기={size}")
    print("  loss 경로:", compact_history)
    print(f"  최종 w={final_w:.6f}, b={final_b:.6f}, loss={final_loss:.6f}")
    print("  최종 prediction:", [round(v, 4) for v in final_pred.tolist()])


### 중간 학습률 비교

0.1은 안정적으로 감소했고 0.5는 크게 발산했음. 그 사이의 0.2와 0.3을 같은 시작값·데이터·loss·12단계로 비교함. 실행 전에 “첫 단계만 줄면 성공인가?”를 판단하고, 전체 경로에서 볼 기준을 세움.

관찰 기준은 전체 loss 방향, 중간 반등, 마지막/처음 비율, 값이 유한한지임. 아래 출력은 y축을 잘라 발산을 숨기는 그림 대신 모든 step의 실제 값을 compact table로 보여줌.


In [ ]:
midpoint_sizes = [0.2, 0.3]
midpoint_histories = {size: run_step_size_trial(size, steps=12)[0] for size in midpoint_sizes}

print("step | lr=0.2 loss | lr=0.3 loss")
print("-----+-------------+------------")
for step in range(12):
    print(f"{step:>4} | {midpoint_histories[0.2][step]:>11.6f} | {midpoint_histories[0.3][step]:>11.6f}")

for size, history in midpoint_histories.items():
    monotonic = all(next_value <= current_value for current_value, next_value in zip(history, history[1:]))
    ratio = history[-1] / history[0]
    print(f"lr={size}: 전체 단조 감소={monotonic}, 마지막/처음={ratio:.6f}")


표를 보고 0.2와 0.3을 `안정적 감소 / 진동하며 감소 / 발산` 중 하나로 분류하고, 반드시 두 개 이상의 step 값을 근거로 씀.

현재 실행에서 `lr=0.3`은 첫 기록 17 이후 loss가 매 기록 증가하며 불안정하게 발산함. 초기 감소 구간은 관찰되지 않음. `lr=0.2`의 실제 경로도 같은 표의 수치로 분류함.

이 실험은 현재 작은 데이터와 수동 gradient descent에서 학습률이 경로를 바꾼다는 근거임. 모든 모델에 같은 최적 학습률이 적용된다는 결론은 내리지 않음.


![learning-rate comparison](assets/10_learning_rate_comparison.svg)

### 결과 뒤에 이름 붙이기

parameter update에서 한 번에 움직이는 크기를 **learning rate, 학습률**이라고 부름.

- 너무 작음: 같은 방향으로 가도 진전이 매우 느림
- 적절함: 손실을 빠르게 낮추는 경로를 만듦
- 너무 큼: 좋은 지점을 넘어가며 손실이 커질 수 있음

한 번의 작은 실험이 모든 문제의 최적 학습률을 증명하지는 않음.


### 추가 실습 · 한 값만 바꿔 관찰하기

`my_step_size`에 새 값을 넣고 loss 경로가 단조롭게 감소하는지, 느린지, 커지는지 분류함. 다른 조건은 바꾸지 않음.


In [ ]:
my_step_size = None  # 예: 수업 중 선택한 값으로 바꿈

if my_step_size is None:
    print("관찰할 크기를 선택함")
else:
    my_history, *_ = run_step_size_trial(my_step_size)
    print([round(value, 6) for value in my_history])


### 전체 흐름 코드 읽기 — 처음 보는 변수명에서 역할 찾기

아래 코드는 오늘과 같은 수동 학습 구조를 사용하지만 변수명이 모두 바뀌어 있음. 실행하기 전에 줄 번호별 역할을 찾음.

```python
features_new = torch.tensor([1.0, 2.0])
labels_new = torch.tensor([3.0, 5.0])
slope = torch.tensor(0.0, requires_grad=True)
offset = torch.tensor(0.0, requires_grad=True)

estimate = slope * features_new + offset
score = ((estimate - labels_new) ** 2).mean()
score.backward()
with torch.no_grad():
    slope -= 0.1 * slope.grad
    offset -= 0.1 * offset.grad
slope.grad.zero_()
offset.grad.zero_()
check_estimate = slope * features_new + offset
check_score = ((check_estimate - labels_new) ** 2).mean()
```

`Tensor/shape`, `model`, `prediction`, `target`, `loss`, `backward`, `update`, `reset`, `fresh evidence` 역할을 각각 변수 또는 줄과 연결함.


In [ ]:
code_roles = {
    "prediction": "",
    "loss": "",
    "backward": "",
    "update": "",
    "reset": "",
    "fresh_evidence": "",
}

filled_roles = sum(bool(value.strip()) for value in code_roles.values())
print(f"역할 연결: {filled_roles}/{len(code_roles)}개 작성함")
for role, location in code_roles.items():
    print(f"{role:15s} → {location or '아직 연결하지 않음'}")


짝과 비교할 때 변수 이름이 같은지만 보지 않음. 각 줄이 실제로 무엇을 읽고 무엇을 바꾸는지 설명함. 특히 `score.backward()`와 `slope -= ...`의 책임을 분리하고, 마지막 `check_score`가 update 전에 만든 score가 아니라 새 parameter의 fresh evidence인지 확인함.

만약 `check_estimate` 두 줄이 없다면 무엇을 확인할 수 없는지, `zero_()` 두 줄이 없다면 다음 단계에 무엇이 섞이는지도 설명해봄.


### 틀린 문장을 근거로 고치기

아래 문장은 모두 일부러 틀리게 적혀 있음. 먼저 혼자 각 문장을 고치고, 고친 문장 옆에 오늘의 값·shape·출력 중 근거 하나를 연결함. 그 뒤 짝에게 한 문장씩 설명함.

1. shape의 원소 수가 같으면 같은 계산임.
2. `backward()`를 호출하면 parameter가 실제로 바뀜.
3. gradient를 0으로 만들면 학습한 parameter도 0으로 돌아감.
4. MSE가 같으면 모든 샘플의 error pattern도 같음.
5. 학습률은 클수록 더 빨리 loss를 낮춤.
6. loss가 줄었으면 보지 않은 데이터에서도 잘한다고 말할 수 있음.


In [ ]:
my_corrections = {number: {"correction": "", "evidence": ""} for number in range(1, 7)}

completed_corrections = sum(
    bool(item["correction"].strip()) and bool(item["evidence"].strip())
    for item in my_corrections.values()
)
print(f"오개념 교정과 근거 연결: {completed_corrections}/6개 완료함")


개인 수정 4분 → 각 문장에 근거 연결 2분 → 짝에게 세 문장씩 설명 순서로 진행함.

짝은 표현이 같은지보다 근거가 실제 주장과 연결되는지 확인함. 서로의 설명을 들은 뒤 가장 근거가 약했던 문장 하나를 다시 고침. 이 장면에서는 최종 여섯 질문을 반복하지 않으며, 다음 전체 지도 회수 뒤 한 번만 자기 말 설명을 수행함.


## Closing · 오늘의 전체 지도를 자기 말로 회수함

![Day 01 전체 학습 지도](assets/01_learning_loop.svg)


### 지도 callback

- Tensor: 데이터와 parameter를 값·shape·dtype·device와 함께 표현함
- Model: 입력에서 prediction을 만드는 계산 규칙임
- Prediction: 현재 parameter가 만든 결과임
- Loss: prediction이 target과 얼마나 다른지 측정함
- Gradient: parameter 변화에 loss가 어느 방향으로 얼마나 민감한지 나타냄
- Update: gradient를 사용해 parameter 값을 실제로 바꿈

그리고 바뀐 parameter로 fresh prediction을 만들어 다시 확인함.


### 자기 말 설명 — 먼저 쓰고, 근거를 붙이고, 설명해보셈

다음 문장을 노트를 보지 않고 먼저 완성해봄.

1. shape가 중요한 이유는 …
2. 실행됐지만 의도한 계산이 아닐 수 있는 예는 …
3. `backward()`가 하는 일과 하지 않는 일은 …
4. parameter가 실제로 바뀌는 순간은 …
5. gradient reset과 fresh forward가 각각 필요한 이유는 …
6. 학습률을 마지막 loss 하나가 아니라 전체 경로로 봐야 하는 이유는 …


### Day 2로 이어지는 질문

오늘은 `w`와 `b`를 직접 관리하고, parameter update와 gradient reset을 직접 수행함.

> parameter가 수백만 개라면 이 책임을 어떻게 일관되게 관리할 수 있을까?

Day 2에서는 parameter를 모델에 등록하고, 반복 update를 optimizer에 맡기며, 더 많은 입력과 더 복잡한 관계를 다루는 구조로 확장함.


---

수업 뒤에는 `pytorch_Q.ipynb`에서 P01부터 순서대로 실행함. 답과 해설은 별도 Practice A에 있으며 먼저 열지 않음.